# Semana 4: Diseño de Validación Cruzada y Control Experimental

**Dataset:** Personas usuarias de Internet por grupo etario, países seleccionados, 2016–2022  
**Objetivo:** Establecer un protocolo experimental robusto basado en validación cruzada temporal para guiar la optimización de hiperparámetros y selección de modelos, evitando la fuga de información (data leakage).

---

## 1. Justificación del Diseño de Validación Cruzada y Control Anti-Leakage

### El Problema del K-Fold Tradicional en Datos Temporales
En conjuntos de datos con una clara estructura temporal, el uso de validación cruzada aleatoria clásica (*K-Fold*) introduce graves problemas de **fuga de información (data leakage)** y evaluaciones sesgadas:
1. **Fuga Temporal (Look-ahead bias):** Si asignamos aleatoriamente filas a los pliegues, un modelo entrenado con datos de 2020 se utilizará para predecir observaciones del pasado (como 2017). En problemas reales, solo podemos predecir el futuro a partir de datos del pasado. Entrenar con "datos del futuro" infla artificialmente las métricas del modelo.
2. **Autocorrelación Temporal:** En datos de tipo panel (donde seguimos a los mismos países y grupos etarios año con año), la observación de `(País A, Grupo B, Año 2017)` está altamente correlacionada con `(País A, Grupo B, Año 2018)`. Si una va al entrenamiento y otra a la validación por selección aleatoria, el modelo simplemente aprenderá a interpolar un vecino muy cercano en lugar de aprender patrones reales de generalización.

### Solución: Validación Cruzada de Ventana Expandida Temporal (*Temporal Expanding Window CV*)
Para abordar estos problemas, implementamos un esquema de validación cruzada temporal adaptado al formato anual de nuestro dataset. La lógica consiste en definir folds basados exclusivamente en la cronología anual del conjunto de entrenamiento (**2016–2020**):
- **Fold 1:** Entrena con **2016**, Valida en **2017**
- **Fold 2:** Entrena con **2016–2017**, Valida en **2018**
- **Fold 3:** Entrena con **2016–2018**, Valida en **2019**
- **Fold 4:** Entrena con **2016–2019**, Valida en **2020**

### Control de Separación (Entrenamiento, Validación y Prueba)
- **Conjunto de Entrenamiento & Validación (CV):** Observaciones de **2016 a 2020** (271 observaciones). Se utiliza para ajustar modelos y seleccionar hiperparámetros a través de los 4 folds temporales descritos arriba.
- **Conjunto de Prueba Final (Hold-out Test):** Observaciones de **2021 a 2022** (70 observaciones). Este conjunto permanece completamente aislado durante todo el proceso de optimización. Se utiliza únicamente al final para calcular la estimación definitiva del error de generalización. Se conserva este split temporal como referencia sólida de acuerdo con la lógica establecida en la Semana 3 para permitir comparabilidad direta.

## 2. Configuración y Carga de Datos

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Carga del dataset preprocesado
df = pd.read_csv('../outputs/datos_transformados_semana2.csv')
print(f"Dimensiones del dataset: {df.shape}")
df.head()

Dimensiones del dataset: (340, 9)


,pais,anio,years_since_2016,porcentaje_internet,grupo_18 a 25 anos de edad,grupo_26 a 50 anos de edad,grupo_51 a 65 anos,grupo_66 anos en adelante,grupo_edad de medicion a 17 anos
0,Argentina,2016,0,76,0,0,0,0,1
1,Argentina,2016,0,86,1,0,0,0,0
2,Argentina,2016,0,82,0,1,0,0,0
3,Argentina,2016,0,61,0,0,1,0,0
4,Argentina,2016,0,29,0,0,0,1,0


In [2]:
# Separar variables explicativas (X) y objetivo (y)
y = df['porcentaje_internet'].copy()
X = df.drop('porcentaje_internet', axis=1).copy()

# Codificar país (LabelEncoder) como en la Semana 3
le_pais = LabelEncoder()
X_encoded = X.copy()
X_encoded['pais'] = le_pais.fit_transform(X_encoded['pais'])

# Definición de máscaras para splits de desarrollo y prueba
train_years = [2016, 2017, 2018, 2019, 2020]
test_years  = [2021, 2022]

train_mask = df['anio'].isin(train_years)
test_mask  = df['anio'].isin(test_years)

X_train = X_encoded[train_mask].reset_index(drop=True)
X_test  = X_encoded[test_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_test  = y[test_mask].reset_index(drop=True)

# Mantener 'anio' en X_train para identificar las particiones del CV,
# pero generar conjuntos finales sin la variable redundante 'anio'
X_train_final = X_train.drop(columns=['anio'])
X_test_final  = X_test.drop(columns=['anio'])

print(f"Conjunto de desarrollo (2016-2020): {X_train_final.shape[0]} observaciones")
print(f"Conjunto de prueba     (2021-2022): {X_test_final.shape[0]} observaciones")

Conjunto de desarrollo (2016-2020): 270 observaciones
Conjunto de prueba     (2021-2022): 70 observaciones


## 3. Implementación de Validación Cruzada Temporal

In [3]:
class TemporalExpandingWindowCV:
    """
    Implementa validación cruzada temporal por ventana expandida para datos anuales.
    Garantiza que la validación se haga sobre un año específico, utilizando únicamente 
    los años anteriores como datos de entrenamiento.
    """
    def __init__(self, train_years=[2016, 2017, 2018, 2019, 2020]):
        self.train_years = sorted(train_years)
        self.n_splits = len(self.train_years) - 1

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError("Se requiere pasar el array de años en el parámetro 'groups'.")
        
        groups = np.asarray(groups)
        indices = np.arange(len(groups))
        
        for i in range(1, len(self.train_years)):
            train_val_years = self.train_years[:i]
            val_year = self.train_years[i]
            
            # Máscaras de índices posicionales
            train_idx = indices[np.isin(groups, train_val_years)]
            val_idx = indices[groups == val_year]
            
            if len(train_idx) > 0 and len(val_idx) > 0:
                yield train_idx, val_idx

### Verificación de los Pliegues Generados

In [4]:
cv = TemporalExpandingWindowCV()
groups = X_train['anio']  # Los años del conjunto de desarrollo para definir folds

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_final, y_train, groups=groups)):
    years_train = sorted(groups[train_idx].unique())
    years_val = sorted(groups[val_idx].unique())
    print(f"Fold {fold + 1}:")
    print(f"  Entrenamiento -> Años: {years_train} | N: {len(train_idx)}")
    print(f"  Validación    -> Años: {years_val}   | N: {len(val_idx)}")

Fold 1:
  Entrenamiento -> Años: [np.int64(2016)] | N: 60
  Validación    -> Años: [np.int64(2017)]   | N: 55
Fold 2:
  Entrenamiento -> Años: [np.int64(2016), np.int64(2017)] | N: 115
  Validación    -> Años: [np.int64(2018)]   | N: 60
Fold 3:
  Entrenamiento -> Años: [np.int64(2016), np.int64(2017), np.int64(2018)] | N: 175
  Validación    -> Años: [np.int64(2019)]   | N: 60
Fold 4:
  Entrenamiento -> Años: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)] | N: 235
  Validación    -> Años: [np.int64(2020)]   | N: 35


## 4. Protocolo Experimental Estandarizado

Definimos una función de evaluación que encapsula todo el proceso de validación cruzada temporal y el cálculo de métricas para un modelo. Esto asegura que **todos los modelos** (incluyendo optimizaciones posteriores) sean evaluados bajo el mismo protocolo experimental sin desviaciones.

In [5]:
def evaluate_model_protocol(model, X_train, y_train, groups, X_test, y_test, name="Modelo"):
    """
    Evalúa un modelo usando el protocolo experimental oficial:
    1. Validación Cruzada Temporal (ventana expandida 2016-2020)
    2. Entrenamiento completo en 2016-2020 y evaluación sobre Prueba (2021-2022)
    """
    cv = TemporalExpandingWindowCV()
    
    cv_maes = []
    cv_rmses = []
    cv_r2s = []
    
    # 1. Validación Cruzada Temporal
    print(f"Iniciando validación cruzada temporal para: {name}...")
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train, groups=groups)):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        
        mae = mean_absolute_error(y_va, preds)
        rmse = np.sqrt(mean_squared_error(y_va, preds))
        r2 = r2_score(y_va, preds)
        
        cv_maes.append(mae)
        cv_rmses.append(rmse)
        cv_r2s.append(r2)
        
        val_year = groups[val_idx].iloc[0]
        print(f"  Fold {fold+1} (Val Año {val_year}): MAE={mae:.2f} pp | RMSE={rmse:.2f} pp | R²={r2:.3f}")
        
    # Resumen de CV
    cv_summary = {
        'CV_MAE_mean': np.mean(cv_maes),
        'CV_MAE_std': np.std(cv_maes),
        'CV_RMSE_mean': np.mean(cv_rmses),
        'CV_RMSE_std': np.std(cv_rmses),
        'CV_R2_mean': np.mean(cv_r2s),
        'CV_R2_std': np.std(cv_r2s)
    }
    
    print(f"\nResumen CV:")
    print(f"  MAE:  {cv_summary['CV_MAE_mean']:.2f} ± {cv_summary['CV_MAE_std']:.2f} pp")
    print(f"  RMSE: {cv_summary['CV_RMSE_mean']:.2f} ± {cv_summary['CV_RMSE_std']:.2f} pp")
    print(f"  R²:   {cv_summary['CV_R2_mean']:.3f} ± {cv_summary['CV_R2_std']:.3f}")
    
    # 2. Evaluación en Test Final
    model.fit(X_train, y_train)
    test_preds = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_preds)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    test_r2 = r2_score(y_test, test_preds)
    
    print(f"\nEvaluación en Conjunto de Prueba Final (2021-2022):")
    print(f"  MAE:  {test_mae:.2f} pp")
    print(f"  RMSE: {test_rmse:.2f} pp")
    print(f"  R²:   {test_r2:.3f}")
    print("="*50 + "\n")
    
    return cv_summary, {'Test_MAE': test_mae, 'Test_RMSE': test_rmse, 'Test_R2': test_r2}

## 5. Evaluación de Modelos de Referencia (Baseline)

Sometemos los tres modelos baseline de la Semana 3 a este protocolo experimental oficial.

In [6]:
baseline_models = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results_cv = {}
results_test = {}

for name, model in baseline_models.items():
    cv_res, test_res = evaluate_model_protocol(
        model, 
        X_train_final, 
        y_train, 
        groups=groups, 
        X_test=X_test_final, 
        y_test=y_test,
        name=name
    )
    results_cv[name] = cv_res
    results_test[name] = test_res

Iniciando validación cruzada temporal para: Regresión Lineal...
  Fold 1 (Val Año 2017): MAE=14.91 pp | RMSE=17.47 pp | R²=0.576
  Fold 2 (Val Año 2018): MAE=13.63 pp | RMSE=16.71 pp | R²=0.565
  Fold 3 (Val Año 2019): MAE=12.29 pp | RMSE=15.28 pp | R²=0.629
  Fold 4 (Val Año 2020): MAE=9.57 pp | RMSE=12.25 pp | R²=0.698

Resumen CV:
  MAE:  12.60 ± 1.98 pp
  RMSE: 15.43 ± 2.00 pp
  R²:   0.617 ± 0.053

Evaluación en Conjunto de Prueba Final (2021-2022):
  MAE:  8.35 pp
  RMSE: 10.60 pp
  R²:   0.732

Iniciando validación cruzada temporal para: Random Forest...


  Fold 1 (Val Año 2017): MAE=9.09 pp | RMSE=12.46 pp | R²=0.785


  Fold 2 (Val Año 2018): MAE=4.60 pp | RMSE=5.71 pp | R²=0.949


  Fold 3 (Val Año 2019): MAE=5.34 pp | RMSE=6.64 pp | R²=0.930


  Fold 4 (Val Año 2020): MAE=6.08 pp | RMSE=7.42 pp | R²=0.889

Resumen CV:
  MAE:  6.28 ± 1.71 pp
  RMSE: 8.06 ± 2.61 pp
  R²:   0.888 ± 0.064



Evaluación en Conjunto de Prueba Final (2021-2022):
  MAE:  5.81 pp
  RMSE: 6.92 pp
  R²:   0.886

Iniciando validación cruzada temporal para: Gradient Boosting...
  Fold 1 (Val Año 2017): MAE=7.66 pp | RMSE=11.98 pp | R²=0.801


  Fold 2 (Val Año 2018): MAE=4.58 pp | RMSE=5.87 pp | R²=0.946
  Fold 3 (Val Año 2019): MAE=5.76 pp | RMSE=7.38 pp | R²=0.913


  Fold 4 (Val Año 2020): MAE=6.84 pp | RMSE=8.06 pp | R²=0.869

Resumen CV:
  MAE:  6.21 ± 1.16 pp
  RMSE: 8.32 ± 2.26 pp
  R²:   0.882 ± 0.055

Evaluación en Conjunto de Prueba Final (2021-2022):
  MAE:  5.48 pp
  RMSE: 6.92 pp
  R²:   0.886



## 6. Comparación y Análisis del Baseline

Resumimos las métricas obtenidas tanto en la validación cruzada temporal como en la prueba final para entender el comportamiento de los modelos.

In [7]:
summary_rows = []
for name in baseline_models.keys():
    summary_rows.append({
        'Modelo': name,
        'CV MAE (pp)': f"{results_cv[name]['CV_MAE_mean']:.2f} ± {results_cv[name]['CV_MAE_std']:.2f}",
        'CV RMSE (pp)': f"{results_cv[name]['CV_RMSE_mean']:.2f} ± {results_cv[name]['CV_RMSE_std']:.2f}",
        'CV R²': f"{results_cv[name]['CV_R2_mean']:.3f} ± {results_cv[name]['CV_R2_std']:.3f}",
        'Test MAE (pp)': f"{results_test[name]['Test_MAE']:.2f}",
        'Test RMSE (pp)': f"{results_test[name]['Test_RMSE']:.2f}",
        'Test R²': f"{results_test[name]['Test_R2']:.3f}"
    })

df_summary = pd.DataFrame(summary_rows).set_index('Modelo')
df_summary

,CV MAE (pp),CV RMSE (pp),CV R²,Test MAE (pp),Test RMSE (pp),Test R²
Modelo,,,,,,
Regresión Lineal,12.60 ± 1.98,15.43 ± 2.00,0.617 ± 0.053,8.35,10.60,0.732
Random Forest,6.28 ± 1.71,8.06 ± 2.61,0.888 ± 0.064,5.81,6.92,0.886
Gradient Boosting,6.21 ± 1.16,8.32 ± 2.26,0.882 ± 0.055,5.48,6.92,0.886


### Análisis de Hallazgos Clave

1. **Métricas de CV vs. Métricas de Prueba (Test):**
   - Observamos que para todos los modelos, el rendimiento promedio de CV es ligeramente peor (mayor error) que el obtenido en el conjunto de prueba (2021-2022). Por ejemplo, el Gradient Boosting tiene un MAE de **6.21 pp** en CV frente a un MAE de **5.48 pp** en Prueba.
   - Esto se explica porque en los primeros folds del CV temporal, el modelo entrena con muy pocos datos. Específicamente, en el **Fold 1** (valida 2017) el modelo solo entrena con el año 2016 (60 observaciones), lo que naturalmente eleva el error promedio. En cambio, para predecir el conjunto de prueba (2021-2022), el modelo se entrena con la historia completa de desarrollo (2016-2020, 271 observaciones), logrando un ajuste muy superior y mejor generalización.
   
2. **El Fold 1 como baseline extremo:**
   - En el Fold 1 (entrenamiento con 2016, validación en 2017), el modelo lineal obtiene un MAE de 14.91 pp, y el Gradient Boosting 7.66 pp. Esto confirma la sensibilidad al volumen de datos en problemas temporales. La métrica consolidada de validación cruzada temporal representa una estimación conservadora (pesimista) pero **segura** del comportamiento del modelo, eliminando por completo cualquier optimismo artificial.

3. **Gradient Boosting como Mejor Modelo Baseline:**
   - Gradient Boosting y Random Forest muestran un desempeño muy competitivo en R² (alrededor de 0.88), superando por mucho a la Regresión Lineal (~0.62 en CV). Gradient Boosting lidera en regularidad del error absoluto medio (MAE CV de 6.21 pp y Test de 5.48 pp).
   
Este protocolo experimental queda establecido como el estándar del equipo para la siguiente etapa de optimización de hiperparámetros.

## 7. Persona 2: Afinamiento de Hiperparámetros

**Decisión sobre Regresión Lineal:** CV MAE = 12.60 ± 1.98 pp — más del doble que RF (6.28) y GB (6.21). La brecha es suficiente para descartar el tuning. Se mantiene como baseline fijo para la comparación final de Persona 3.

**Enfoque:** `GridSearchCV` con `TemporalExpandingWindowCV` de Persona 1. El test set (2021-2022) **no se usa** para elegir hiperparámetros — solo para la evaluación final de Persona 3.

In [8]:
from sklearn.model_selection import GridSearchCV

# constantes de referencia para tdd postcondiciones
BASELINE_RF_CV_MAE = 6.28
BASELINE_GB_CV_MAE = 6.21

# aliases para cálculo de mejora en test
test_rf_baseline = results_test['Random Forest']
test_gb_baseline = results_test['Gradient Boosting']

# tdd: precondiciones — artefactos de persona 1 deben estar disponibles
assert X_train_final.shape == (270, 7), f"forma esperada (270, 7), obtenida {X_train_final.shape}"
assert X_test_final.shape == (70, 7), f"forma esperada (70, 7), obtenida {X_test_final.shape}"
assert len(groups) == 270, f"groups debe tener 270 elementos, tiene {len(groups)}"
assert 'anio' not in X_train_final.columns, "anio no debe estar en X_train_final"
print("Precondiciones OK — artefactos de Persona 1 verificados")

Precondiciones OK — artefactos de Persona 1 verificados


### 7.1 Tuning: Random Forest Regressor

| Hiperparámetro | Valores | Razón |
|---|---|---|
| `n_estimators` | [50, 100, 200] | cantidad de árboles |
| `max_depth` | [None, 5, 10] | control de overfitting |
| `min_samples_leaf` | [1, 2, 4] | suavizado de hojas |
| `max_features` | [1.0, 'sqrt', 'log2'] | incluye default (1.0=todas) + variantes reducidas |

**Total:** 81 combinaciones × 4 folds = 324 fits. Se incluye `1.0` para garantizar que el grid cubra la configuración base del modelo (sklearn default para regresión).

In [9]:
# búsqueda de hiperparámetros para random forest
gs_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid={
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [1.0, 'sqrt', 'log2']
    },
    cv=TemporalExpandingWindowCV(),
    scoring='neg_mean_absolute_error',
    refit=True,
    n_jobs=-1
)
gs_rf.fit(X_train_final, y_train, groups=groups)
print(f"Mejores parámetros RF: {gs_rf.best_params_}")
print(f"CV MAE optimizado RF: {-gs_rf.best_score_:.2f} pp")

Mejores parámetros RF: {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1, 'n_estimators': 200}
CV MAE optimizado RF: 6.26 pp


In [10]:
# tdd: postcondiciones rf — el tuning debe mejorar o igualar el baseline
assert len(gs_rf.best_params_) > 0, "gs_rf debe tener best_params_"
assert (-gs_rf.best_score_) <= BASELINE_RF_CV_MAE, (
    f"RF optimizado ({-gs_rf.best_score_:.2f}) no debe ser peor que baseline ({BASELINE_RF_CV_MAE})"
)
print(f"RF optimizado: CV MAE = {-gs_rf.best_score_:.2f} pp (baseline: {BASELINE_RF_CV_MAE} pp)")

RF optimizado: CV MAE = 6.26 pp (baseline: 6.28 pp)


In [11]:
cv_rf_opt, test_rf_opt = evaluate_model_protocol(
    gs_rf.best_estimator_,
    X_train_final, y_train,
    groups=groups,
    X_test=X_test_final, y_test=y_test,
    name="Random Forest Optimizado"
)

print(f"Mejora CV MAE vs baseline: {BASELINE_RF_CV_MAE - cv_rf_opt['CV_MAE_mean']:.2f} pp")
print(f"Mejora Test MAE vs baseline: {test_rf_baseline['Test_MAE'] - test_rf_opt['Test_MAE']:.2f} pp")

Iniciando validación cruzada temporal para: Random Forest Optimizado...


  Fold 1 (Val Año 2017): MAE=8.96 pp | RMSE=12.56 pp | R²=0.781


  Fold 2 (Val Año 2018): MAE=4.64 pp | RMSE=5.81 pp | R²=0.947


  Fold 3 (Val Año 2019): MAE=5.35 pp | RMSE=6.70 pp | R²=0.929


  Fold 4 (Val Año 2020): MAE=6.09 pp | RMSE=7.49 pp | R²=0.887

Resumen CV:
  MAE:  6.26 ± 1.64 pp
  RMSE: 8.14 ± 2.62 pp
  R²:   0.886 ± 0.064



Evaluación en Conjunto de Prueba Final (2021-2022):
  MAE:  5.67 pp
  RMSE: 6.76 pp
  R²:   0.891

Mejora CV MAE vs baseline: 0.02 pp
Mejora Test MAE vs baseline: 0.14 pp


### 7.2 Tuning: Gradient Boosting Regressor

| Hiperparámetro | Valores | Razón |
|---|---|---|
| `n_estimators` | [50, 100, 200] | boosting rounds |
| `max_depth` | [2, 3, 5] | profundidad de árboles débiles |
| `learning_rate` | [0.05, 0.1, 0.2] | tasa de corrección |
| `min_samples_leaf` | [1, 2, 4] | control de hojas |

**Total:** 81 combinaciones × 4 folds = 324 fits. Mismo protocolo que RF.

In [12]:
# búsqueda de hiperparámetros para gradient boosting
gs_gb = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid={
        'n_estimators': [50, 100, 200],
        'max_depth': [2, 3, 5],
        'learning_rate': [0.05, 0.1, 0.2],
        'min_samples_leaf': [1, 2, 4]
    },
    cv=TemporalExpandingWindowCV(),
    scoring='neg_mean_absolute_error',
    refit=True,
    n_jobs=-1
)
gs_gb.fit(X_train_final, y_train, groups=groups)
print(f"Mejores parámetros GB: {gs_gb.best_params_}")
print(f"CV MAE optimizado GB: {-gs_gb.best_score_:.2f} pp")

Mejores parámetros GB: {'learning_rate': 0.2, 'max_depth': 5, 'min_samples_leaf': 2, 'n_estimators': 200}
CV MAE optimizado GB: 5.33 pp


In [13]:
# tdd: postcondiciones gb — el tuning debe mejorar o igualar el baseline
assert len(gs_gb.best_params_) > 0, "gs_gb debe tener best_params_"
assert (-gs_gb.best_score_) <= BASELINE_GB_CV_MAE, (
    f"GB optimizado ({-gs_gb.best_score_:.2f}) no debe ser peor que baseline ({BASELINE_GB_CV_MAE})"
)
print(f"GB optimizado: CV MAE = {-gs_gb.best_score_:.2f} pp (baseline: {BASELINE_GB_CV_MAE} pp)")

GB optimizado: CV MAE = 5.33 pp (baseline: 6.21 pp)


In [14]:
cv_gb_opt, test_gb_opt = evaluate_model_protocol(
    gs_gb.best_estimator_,
    X_train_final, y_train,
    groups=groups,
    X_test=X_test_final, y_test=y_test,
    name="Gradient Boosting Optimizado"
)

print(f"Mejora CV MAE vs baseline: {BASELINE_GB_CV_MAE - cv_gb_opt['CV_MAE_mean']:.2f} pp")
print(f"Mejora Test MAE vs baseline: {test_gb_baseline['Test_MAE'] - test_gb_opt['Test_MAE']:.2f} pp")

Iniciando validación cruzada temporal para: Gradient Boosting Optimizado...
  Fold 1 (Val Año 2017): MAE=7.19 pp | RMSE=12.01 pp | R²=0.800
  Fold 2 (Val Año 2018): MAE=4.29 pp | RMSE=5.41 pp | R²=0.954


  Fold 3 (Val Año 2019): MAE=4.36 pp | RMSE=5.34 pp | R²=0.955
  Fold 4 (Val Año 2020): MAE=5.49 pp | RMSE=7.29 pp | R²=0.893

Resumen CV:
  MAE:  5.33 ± 1.17 pp
  RMSE: 7.51 ± 2.71 pp
  R²:   0.900 ± 0.063



Evaluación en Conjunto de Prueba Final (2021-2022):
  MAE:  4.97 pp
  RMSE: 6.11 pp
  R²:   0.911

Mejora CV MAE vs baseline: 0.88 pp
Mejora Test MAE vs baseline: 0.51 pp


### 9. Exportación de Hiperparámetros para Persona 3

Los mejores hiperparámetros encontrados para RF y GB se exportan a `outputs/mejores_hiperparametros_semana4.csv`.
Esto permite que Persona 3 referencie la configuración exacta de cada modelo optimizado al construir la tabla comparativa final.

In [15]:
import os

# consolidar mejores hiperparámetros para que persona 3 pueda referenciarlos
rows = []
for param, value in gs_rf.best_params_.items():
    rows.append({'modelo': 'Random Forest Optimizado', 'hiperparametro': param, 'valor': value})
for param, value in gs_gb.best_params_.items():
    rows.append({'modelo': 'Gradient Boosting Optimizado', 'hiperparametro': param, 'valor': value})

df_best_params = pd.DataFrame(rows)
df_best_params.to_csv('../outputs/mejores_hiperparametros_semana4.csv', index=False)
print(f"Hiperparámetros exportados: {len(df_best_params)} filas")
print(df_best_params.to_string(index=False))

Hiperparámetros exportados: 8 filas
                      modelo   hiperparametro  valor
    Random Forest Optimizado        max_depth    NaN
    Random Forest Optimizado     max_features    1.0
    Random Forest Optimizado min_samples_leaf    1.0
    Random Forest Optimizado     n_estimators  200.0
Gradient Boosting Optimizado    learning_rate    0.2
Gradient Boosting Optimizado        max_depth    5.0
Gradient Boosting Optimizado min_samples_leaf    2.0
Gradient Boosting Optimizado     n_estimators  200.0


In [16]:
# tdd: verificar que el csv de hiperparámetros se exportó correctamente
df_params_check = pd.read_csv('../outputs/mejores_hiperparametros_semana4.csv')
assert len(df_params_check) > 0, "el csv no debe estar vacío"
assert set(['modelo', 'hiperparametro', 'valor']).issubset(df_params_check.columns), "faltan columnas"
print(f"CSV verificado: {len(df_params_check)} filas, columnas {list(df_params_check.columns)}")
print(df_params_check.to_string(index=False))

CSV verificado: 8 filas, columnas ['modelo', 'hiperparametro', 'valor']
                      modelo   hiperparametro  valor
    Random Forest Optimizado        max_depth    NaN
    Random Forest Optimizado     max_features    1.0
    Random Forest Optimizado min_samples_leaf    1.0
    Random Forest Optimizado     n_estimators  200.0
Gradient Boosting Optimizado    learning_rate    0.2
Gradient Boosting Optimizado        max_depth    5.0
Gradient Boosting Optimizado min_samples_leaf    2.0
Gradient Boosting Optimizado     n_estimators  200.0
